# Random Module and Reproducibility

Random numbers are essential in data science:
- **Shuffling** data before splitting into train/test sets
- **Sampling** a subset from a large dataset
- **Simulating** scenarios (Monte Carlo methods)
- **Initializing** weights in machine learning models

NumPy has a powerful random number generator built in.

**What we'll cover:**
1. The modern random API (`default_rng`)
2. Generating random integers and floats
3. Normal distribution (the bell curve)
4. Shuffling and choosing
5. Seeds and reproducibility
6. The old API (and why to avoid it)

In [ ]:
import numpy as np

---

## 1. Creating a Random Number Generator

NumPy's recommended way to generate random numbers is through a **Generator** object created with `np.random.default_rng()`.

Think of it as creating a "random number machine" that you can use throughout your code.

In [ ]:
# Create a random number generator (RNG)
rng = np.random.default_rng()

# Generate a single random float between 0 and 1
print("One random float:", rng.random())

# Generate 5 random floats
print("Five random floats:", rng.random(5))

> **Note:** Every time you run the cell above, you'll get different numbers. We'll learn how to fix that later with **seeds**.

---

## 2. Random Integers

In [ ]:
rng = np.random.default_rng()

# One random integer between 1 and 6 (like rolling a die)
# Note: the upper bound (7) is EXCLUDED
print("Die roll:", rng.integers(1, 7))
print()

# 10 die rolls at once
print("10 rolls:", rng.integers(1, 7, size=10))
print()

# A 3×4 grid of random integers between 0 and 100
grid = rng.integers(0, 101, size=(3, 4))
print("Random grid:")
print(grid)

> **The bounds:** `rng.integers(low, high)` includes `low` but **excludes** `high`. So `rng.integers(1, 7)` gives you 1, 2, 3, 4, 5, or 6.

---

## 3. Random Floats in a Range

In [ ]:
rng = np.random.default_rng()

# Random floats between 0 and 1 (default)
print("Default [0, 1):", rng.random(5))
print()

# Random floats in a custom range using uniform()
# Example: random temperatures between 20°C and 35°C
random_temps = rng.uniform(low=20, high=35, size=7)
print("Random temps:", np.round(random_temps, 1))

---

## 4. Normal (Gaussian) Distribution — The Bell Curve

This is the most important probability distribution in statistics. It shows up everywhere — heights, test scores, measurement errors, stock returns.

A normal distribution is defined by two numbers:
- **Mean (loc)** — the center of the bell curve
- **Standard deviation (scale)** — how wide/narrow the bell is

In [ ]:
rng = np.random.default_rng(42)

# Generate 10,000 exam scores: mean=70, std=10
exam_scores = rng.normal(loc=70, scale=10, size=10000)

# Verify: the actual mean and std should be close to what we specified
print(f"Requested:  mean=70, std=10")
print(f"Got:        mean={np.mean(exam_scores):.2f}, std={np.std(exam_scores):.2f}")
print(f"Min: {np.min(exam_scores):.1f}, Max: {np.max(exam_scores):.1f}")

With 10,000 samples, the actual mean and std are very close to what we requested. The more samples you generate, the closer they'll be.

In [ ]:
# Standard Normal Distribution: mean=0, std=1
# This is so common it has its own shortcut
standard = rng.standard_normal(size=10)
print("Standard normal (mean=0, std=1):")
print(np.round(standard, 3))

---

## 5. Shuffling and Choosing

Two very common operations when working with datasets.

### Shuffling — Rearranging in Random Order

In [ ]:
rng = np.random.default_rng(42)

deck = np.arange(1, 11)   # "Cards" numbered 1-10
print("Original:", deck)

# shuffle() modifies the array in-place (no return value)
rng.shuffle(deck)
print("Shuffled:", deck)

In [ ]:
# If you want a shuffled COPY (original stays unchanged), use permutation()
original = np.arange(1, 11)
shuffled = rng.permutation(original)

print("Original:", original)   # Still [1, 2, 3, ...]
print("Shuffled:", shuffled)   # Randomized copy

### Choosing — Random Selection from an Array

In [ ]:
rng = np.random.default_rng(42)

colors = np.array(["red", "blue", "green", "yellow", "purple"])

# Pick 3 random colors WITHOUT replacement (no repeats)
picked = rng.choice(colors, size=3, replace=False)
print("Without replacement:", picked)

# Pick 8 random colors WITH replacement (repeats allowed)
picked = rng.choice(colors, size=8, replace=True)
print("With replacement:   ", picked)

### Weighted Random Choice

Sometimes you want some options to be more likely than others. Pass a `p` (probability) array.

In [ ]:
rng = np.random.default_rng(42)

# Simulate a loaded die where 6 appears 50% of the time
die_faces = np.array([1, 2, 3, 4, 5, 6])
weights = [0.1, 0.1, 0.1, 0.1, 0.1, 0.5]   # Must add up to 1.0

rolls = rng.choice(die_faces, size=1000, p=weights)

# Check how often each face appeared
print("Results from 1000 rolls of a loaded die:")
for face in die_faces:
    count = np.sum(rolls == face)
    print(f"  Face {face}: {count:3d} times ({count/10:.1f}%)")

---

## 6. Seeds and Reproducibility

Here's a common problem: you write some code that uses random numbers, and every time you run it, you get different results. This makes debugging hard and makes it impossible for someone else to verify your work.

**The solution: set a seed.**

A **seed** is a starting number for the random number generator. The same seed always produces the **exact same sequence** of "random" numbers.

In [ ]:
# WITHOUT a seed: different results every time
rng_a = np.random.default_rng()
rng_b = np.random.default_rng()

print("No seed:")
print("  Run A:", rng_a.integers(0, 100, size=5))
print("  Run B:", rng_b.integers(0, 100, size=5))
print("  (These are different)")

In [ ]:
# WITH a seed: same results every time
rng_a = np.random.default_rng(seed=42)
rng_b = np.random.default_rng(seed=42)

print("With seed=42:")
print("  Run A:", rng_a.integers(0, 100, size=5))
print("  Run B:", rng_b.integers(0, 100, size=5))
print("  (These are identical!)")

> **Which seed number should I use?** It doesn't matter — 42, 0, 123, 999, any integer works. What matters is that you use the **same seed** when you want the same results.
>
> `42` is just a popular choice (it's a reference to The Hitchhiker's Guide to the Galaxy).

### Best Practice: One RNG for Your Whole Script

Create a single generator at the top and reuse it everywhere.

In [ ]:
# Good practice
rng = np.random.default_rng(seed=123)

# All these use the same generator
data = rng.normal(0, 1, size=100)
indices = rng.integers(0, 100, size=10)
shuffled = rng.permutation(50)

print("Data (first 5):", np.round(data[:5], 3))
print("Random indices:", indices[:5])
print("Shuffled (first 5):", shuffled[:5])

---

## 7. The Old API (Legacy)

Older code and tutorials use a different style — calling functions directly on `np.random`. This still works but has some issues, so the `default_rng()` approach is recommended for new code.

In [ ]:
# OLD way — still works, but NOT recommended for new code
np.random.seed(42)
print("Old: random ints:", np.random.randint(0, 100, size=5))
print("Old: random floats:", np.round(np.random.rand(3), 4))
print("Old: normal dist:", np.round(np.random.randn(3), 4))
print()

# NEW way — recommended
rng = np.random.default_rng(42)
print("New: random ints:", rng.integers(0, 100, size=5))
print("New: random floats:", np.round(rng.random(3), 4))
print("New: normal dist:", np.round(rng.standard_normal(3), 4))

**Why is the old way not recommended?**

- `np.random.seed()` uses a **global** state — changing it in one function can affect another
- The new Generator uses a better underlying algorithm
- The new API has clearer naming (`integers` vs `randint`, `random` vs `rand`)

You'll still see the old style in many tutorials and Stack Overflow answers. Just know that `default_rng()` is the better choice going forward.

---

## 8. Practical Examples

### Simulating 10,000 Coin Flips

In [ ]:
rng = np.random.default_rng(42)

# 0 = Tails, 1 = Heads
flips = rng.integers(0, 2, size=10_000)

heads = np.sum(flips == 1)
tails = np.sum(flips == 0)

print(f"Heads: {heads} ({heads / 100:.1f}%)")
print(f"Tails: {tails} ({tails / 100:.1f}%)")
print(f"Ratio: {heads/tails:.3f} (perfect would be 1.000)")

### Train/Test Split — A Very Common Task

In machine learning, you split your data into training and testing sets. Randomization ensures the split is unbiased.

In [ ]:
rng = np.random.default_rng(42)

# Pretend we have 100 data samples (indices 0-99)
n_samples = 100
all_indices = np.arange(n_samples)

# Randomly shuffle the indices
shuffled_indices = rng.permutation(n_samples)

# 80% train, 20% test
split_point = int(0.8 * n_samples)
train_idx = shuffled_indices[:split_point]
test_idx = shuffled_indices[split_point:]

print(f"Total samples: {n_samples}")
print(f"Train set: {len(train_idx)} samples")
print(f"Test set:  {len(test_idx)} samples")
print(f"Train indices (first 10): {sorted(train_idx[:10])}")
print(f"Test indices (first 10):  {sorted(test_idx[:10])}")

---

## Quick Reference

| Function | What it does | Example |
|----------|-------------|---------|
| `np.random.default_rng(seed)` | Create a generator | `rng = np.random.default_rng(42)` |
| `rng.integers(low, high, size)` | Random ints [low, high) | `rng.integers(1, 7, size=10)` |
| `rng.random(size)` | Random floats [0, 1) | `rng.random(5)` |
| `rng.uniform(low, high, size)` | Random floats [low, high) | `rng.uniform(20, 35, size=7)` |
| `rng.normal(loc, scale, size)` | Normal distribution | `rng.normal(70, 10, size=100)` |
| `rng.standard_normal(size)` | Standard normal (0, 1) | `rng.standard_normal(10)` |
| `rng.choice(arr, size, replace)` | Random selection | `rng.choice(colors, 3)` |
| `rng.shuffle(arr)` | Shuffle in-place | `rng.shuffle(deck)` |
| `rng.permutation(arr)` | Shuffled copy | `rng.permutation(100)` |

**Golden rule:** Always set a seed when you need reproducible results.

---

**Up next →** Time to practice! The next notebook has hands-on exercises covering everything we've learned.